# NEXUS Z: train and score on LinuxONE

Runs the NEXUS Z model pipeline inside the Jupyter Lab container on the LinuxONE VM (s390x). Training and scoring both happen here, on CPU. No accelerator is used.

Folders mounted into the container: `~/Nexus-Z` (this repo), `~/data` (datasets, models, cache), `~/shared`.
Run the cells in order. Cells 5 and 6 take several minutes.

In [ ]:
import os, platform, sys, time
os.chdir('/home/ibm-user/Nexus-Z')
os.environ['NEXUSZ_DATA_DIR'] = '/home/ibm-user/data'
sys.path.insert(0, '/home/ibm-user/Nexus-Z')
print(platform.machine(), sys.byteorder + '-endian', os.cpu_count(), 'cpus')
import numpy, pandas, scipy, sklearn, xgboost
print({m.__name__: m.__version__ for m in (numpy, pandas, scipy, sklearn, xgboost)})

## 1. Tests
Runs the model tests that do not need the API libraries (FastAPI and pydantic are not in this image). One test is skipped because it compares against the SHAP package, which is not installed.

In [ ]:
!pip install -q pytest 2>&1 | grep -v -i notice
!python -m pytest ml/tests/test_features.py ml/tests/test_rings.py ml/tests/test_explain.py ml/tests/test_ring_score.py -q 2>&1 | tail -5

## 2. Data
The full HI-Small file and its laundering patterns must be in `~/data/ibm-aml/`.

In [ ]:
!ls -la /home/ibm-user/data/ibm-aml

## 3. Train the model folder on this machine
Builds features for the train, validation and test windows, fits XGBoost, calibrates it, and fits the ring scorer. Writes `models/HI-Small` under the data folder.

In [ ]:
t = time.time()
!python -m ml.bundle HI-Small
print(f'trained in {time.time() - t:.0f}s')

## 4. Does the model trained here agree with the one trained on the laptop?
Compares the model folder committed in the repo against the one just trained, on 300 reference feature rows. Different training runs will not be identical; look at how large the difference is.

In [ ]:
!python -m ml.latency_check check ml/artifacts/HI-Small /home/ibm-user/data/reference.csv
!python -m ml.latency_check check /home/ibm-user/data/models/HI-Small /home/ibm-user/data/reference.csv

## 5. Account-level comparison of feature sets (transaction-only vs graph)

In [ ]:
!python -m ml.ablation HI-Small

## 6. Ring-level evaluation against the IBM laundering patterns

In [ ]:
!python -m ml.ring_eval HI-Small

## 7. Time a full rescore of the last 48 hours with the model trained here

In [ ]:
!python -m ml.latency_check time /home/ibm-user/data/models/HI-Small HI-Slice